# 1. Original C4 + Stack data preparation (CPU only)
Set Accelerator = None and Internet = On. Attach the new `domain-shift-stage1.zip`.
Add and enable a Kaggle Secret named `HF_TOKEN` for your approved Hugging Face account.
This prepares the original dataset; it does not train a model. Keep the reserved-test ZIP separate.
Use local CPU preparation if the candidate pool exceeds Kaggle's RAM/disk or session time.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, zipfile, shutil, tempfile, time
inputs = Path('/kaggle/input')
roots = sorted({p.parent.parent.parent for p in inputs.rglob('pilot.py')
                if p.parent.name == 'domain_shift_forgetting' and p.parent.parent.name == 'src'})
if roots:
    assert len(roots) == 1, f'Attach exactly one stage1 code bundle: {roots}'
    code = roots[0]
else:
    bundles = list(inputs.rglob('domain-shift-stage1.zip'))
    assert len(bundles) == 1, 'Attach domain-shift-stage1.zip as a private dataset'
    code = Path(tempfile.mkdtemp(prefix='stage1-code-'))
    with zipfile.ZipFile(bundles[0]) as z:
        for name in z.namelist():
            assert (code / name).resolve().is_relative_to(code.resolve())
        z.extractall(code)
env = os.environ | {'PYTHONPATH': str(code / 'src'), 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'}
def cli(module, *args):
    subprocess.run([sys.executable, '-m', 'domain_shift_forgetting.' + module,
                    *map(str, args)], env=env, check=True)
print('Code:', code)


In [ ]:
# Do not spend GPU quota on preprocessing.
probe = subprocess.run(['sh', '-c', 'nvidia-smi -L 2>/dev/null'], capture_output=True, text=True)
assert 'GPU 0:' not in probe.stdout, 'Change Kaggle Accelerator to None before preprocessing.'
from kaggle_secrets import UserSecretsClient
env['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'datasets', 'huggingface_hub',
                'pyarrow', 'datasketch', 'tiktoken'], check=True)
base = Path('/kaggle/working/stage1-preparation')
base.mkdir(exist_ok=True)
if not (base / 'upstream/source-evidence.json').exists():
    cli('pilot', 'sources', '--output', base / 'upstream')
access_path = base / 'access/access.json'
if not access_path.exists() or json.loads(access_path.read_text()).get('status') != 'passed':
    cli('pilot_data', 'inspect', '--output', base / 'access')


In [ ]:
# CPU-only. Commits completed source shards to SQLite; rerunning resumes those shards.
prepared = base / 'corpus'
manifest_path = prepared / 'online/manifest.json'
if not manifest_path.exists() or not json.loads(manifest_path.read_text()).get('preparation_status'):
    extra = ['--resume'] if (prepared / 'preparation-settings.json').exists() else []
    cli('pilot_data', 'prepare', '--access', base / 'access/access.json',
        '--output', prepared, '--cache', Path('/kaggle/temp/stage1-cache'), *extra)
if not (base / 'orders/manifest.json').exists():
    cli('pilot', 'orders', '--data-dir', prepared / 'online', '--output', base / 'orders')


In [ ]:
# Only online arrays go to the GPU notebook. Never attach the sealed test archive there.
online_zip = Path('/kaggle/working/stage1-online-inputs.zip')
with zipfile.ZipFile(online_zip, 'w', zipfile.ZIP_STORED) as z:
    for source, prefix in [(prepared / 'online', 'online'), (base / 'orders', 'orders'), (base / 'upstream', 'upstream')]:
        for p in source.rglob('*'):
            if p.is_file():
                z.write(p, str(Path(prefix) / p.relative_to(source)))
sealed_zip = shutil.make_archive('/kaggle/working/stage1-reserved-test-SEALED', 'zip', prepared / 'reserved-test')
print('Download/persist privately and attach to GPU notebook:', online_zip)
print('Download/persist separately; DO NOT attach to GPU notebook:', sealed_zip)
print('Preserve the preparation audit and source receipts:', base)
